# Lab 7 — Large Language Models (LLMs)



## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |


## 1. Setup


In [1]:
%%capture
%pip -q install transformers sentencepiece torch


In [2]:
import os
import warnings

# Keep notebook output clean: hide download bars, cache/path warnings, and library logs
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

warnings.filterwarnings("ignore")

from transformers import pipeline, set_seed
from transformers.utils import logging as transformers_logging
from huggingface_hub.utils import disable_progress_bars

transformers_logging.set_verbosity_error()
disable_progress_bars()

print("Libraries loaded successfully!")


Libraries loaded successfully!


# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.


In [3]:
generator = pipeline("text-generation", model="distilgpt2")

prompt = "Natural language processing is"

output = generator(
    prompt,
    max_new_tokens=30,
    do_sample=False,
    clean_up_tokenization_spaces=False
)

print(output[0]["generated_text"])


Natural language processing is a common feature of the language processing process.























# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.


In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt = "Translate to French: I love natural language processing."

# Convert text into tokens
inputs = tokenizer(prompt, return_tensors="pt")
output = model.generate(**inputs, max_new_tokens=50)

# Convert tokens back to text
print(tokenizer.decode(output[0], skip_special_tokens=True))


Je s'agissant de la technologie de linguage naturelle.


# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.


In [ ]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

text = "This NLP lab is easy and interesting."

result = classifier(text)
print(result)


[{'label': 'POSITIVE', 'score': 0.999778687953949}]


# Tasks

Complete the following tasks.

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

In [6]:
print("""Sentiment classification - Encoder-only
Machine translation - Encoder–Decoder
Open-ended text generation - Decoder-only
Named Entity Recognition - Encoder-only
Text summarization - Encoder–Decoder""")


Sentiment classification - Encoder-only
Machine translation - Encoder–Decoder
Open-ended text generation - Decoder-only
Named Entity Recognition - Encoder-only
Text summarization - Encoder–Decoder


## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

In [7]:
sentences = ["The movie was excellent.", "The lecture was difficult to understand.", "I enjoyed learning about transformers."]


for sentence in sentences:
    result = classifier(sentence)[0]

    print("Sentence:", sentence)
    print("Label:", result["label"])
    print("Score:", result["score"])
    print()

Sentence: The movie was excellent.
Label: POSITIVE
Score: 0.9998608827590942

Sentence: The lecture was difficult to understand.
Label: NEGATIVE
Score: 0.9996652603149414

Sentence: I enjoyed learning about transformers.
Label: POSITIVE
Score: 0.999745786190033



## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.


In [8]:
paragraph = """
Natural language processing is a field of artificial intelligence
that allows computers to understand, interpret, and generate human language.
It is used in applications such as chatbots, translation systems,
sentiment analysis, and text summarization.
"""

prompt = "Summarize: " + paragraph

inputs = tokenizer(prompt, return_tensors="pt")

output = model.generate(
    **inputs,
    max_new_tokens=50
)

summary = tokenizer.decode(output[0], skip_special_tokens=True)

print(summary)

Use natural language processing to understand and interpret the language.



## Task 5 — Short Reflection

Answer in 3–5 sentences:

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**


In [9]:
print("""A decoder-only model is suitable for a chatbot because it can generate
text one token at a time based on the previous conversation. It is designed for text generation,
which allows it to produce natural and relevant responses. It can also use the conversation context to predict
what should come next. This makes decoder-only models effective for interactive conversations such as ChatGPT.""")


A decoder-only model is suitable for a chatbot because it can generate
text one token at a time based on the previous conversation. It is designed for text generation,
which allows it to produce natural and relevant responses. It can also use the conversation context to predict
what should come next. This makes decoder-only models effective for interactive conversations such as ChatGPT.


# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same?

**Hint:** Sampling introduces randomness when choosing the next token.


In [10]:
print("""You may get different outputs because sampling introduces randomness when selecting the next token.
The model assigns probabilities to possible tokens and may choose different ones each time. As a result,
the same prompt can produce different generated text.""")


You may get different outputs because sampling introduces randomness when selecting the next token.
The model assigns probabilities to possible tokens and may choose different ones each time. As a result,
the same prompt can produce different generated text.
